#  Employee Data Engineering & Analysis
**Purpose:** Clean, transform, and analyse company HR data for executive reporting.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded :)")

Libraries loaded :)


## 1. Load Raw Data

In [4]:
df_raw = pd.read_excel('Employee Sample Data.xlsx')
print(f"Rows: {df_raw.shape[0]:,}  |  Columns: {df_raw.shape[1]}")
print("Column names:", df_raw.columns.tolist())
df_raw.head(5)

Rows: 1,000  |  Columns: 14
Column names: ['EEID', 'Full Name', 'Job Title', 'Department', 'Business Unit', 'Gender', 'Ethnicity', 'Age', 'Hire Date', 'Annual Salary', 'Bonus %', 'Country', 'City', 'Exit Date']


,EEID,Full Name,Job Title,Department,Business Unit,Gender,Ethnicity,Age,Hire Date,Annual Salary,Bonus %,Country,City,Exit Date
0,E02387,Emily Davis,Sr. Manger,IT,Research & Development,Female,Black,55.0,2016-04-08,141604.0,0.15,United States,Seattle,2021-10-16
1,E04105,Theodore Dinh,Technical Architect,IT,Manufacturing,Male,Asian,59.0,1997-11-29,99975.0,0.00,China,Chongqing,NaT
2,E02572,Luna Sanders,Director,Finance,Speciality Products,Female,Caucasian,50.0,2006-10-26,163099.0,0.20,United States,Chicago,NaT
3,E02832,Penelope Jordan,Computer Systems Manager,IT,Manufacturing,Female,Caucasian,26.0,2019-09-27,84913.0,0.07,United States,Chicago,NaT
4,E01639,Austin Vo,Sr. Analyst,Finance,Manufacturing,Male,Asian,55.0,1995-11-20,95409.0,0.00,United States,Phoenix,NaT


## 2. Data Quality Assessment

In [5]:
# Missing values report
missing = df_raw.isnull().sum()
pct_missing = (missing / len(df_raw) * 100).round(1)
quality_report = pd.DataFrame({'Missing Count': missing, 'Missing %': pct_missing})
quality_report = quality_report[quality_report['Missing Count'] > 0]
print("=== Missing Values ===")
print(quality_report.to_string())
print(f'\n')
print(f"Duplication: {df_raw.duplicated().sum()} duplicate rows")
print(f'\n')
print(f"Data Types:")
print(df_raw.dtypes.to_string())

=== Missing Values ===
               Missing Count  Missing %
Full Name                  2        0.2
Job Title                  1        0.1
Department                 2        0.2
Gender                     1        0.1
Ethnicity                  7        0.7
Age                        6        0.6
Hire Date                  7        0.7
Annual Salary             11        1.1
Bonus %                    8        0.8
Country                    2        0.2
City                       2        0.2
Exit Date                915       91.5


Duplication: 0 duplicate rows


Data Types:
EEID                     object
Full Name                object
Job Title                object
Department               object
Business Unit            object
Gender                   object
Ethnicity                object
Age                     float64
Hire Date        datetime64[ns]
Annual Salary           float64
Bonus %                 float64
Country                  object
City                     ob

## 3. Data Cleaning & Engineering

In [6]:
df = df_raw.copy()

#Type casting
df['Annual Salary'] = pd.to_numeric(df['Annual Salary'], errors='coerce')
df['Bonus %']       = pd.to_numeric(df['Bonus %'], errors='coerce')
df['Age']           = pd.to_numeric(df['Age'], errors='coerce')
df['Hire Date']     = pd.to_datetime(df['Hire Date'],  errors='coerce')
df['Exit Date']     = pd.to_datetime(df['Exit Date'],  errors='coerce')

#feature eng.
today = pd.Timestamp('2026-04-29')
df['Tenure Years']    = ((df['Exit Date'].fillna(today) - df['Hire Date']).dt.days/365).round(1)#Tenure years = how long an employee has worked in a company
df['Is Active']       = df['Exit Date'].isna()#still work
df['Hire Year']       = df['Hire Date'].dt.year
df['Hire Quarter']    = df['Hire Date'].dt.to_period('Q').astype(str)#company hiring seasons
#to further work if i want to build ML model to predict the salary
df['Age Band']        = pd.cut(df['Age'], bins=[24,34,44,54,65] ,labels=['25–34','35–44','45–54','55–65'])
df['Salary Band']     = pd.cut(df['Annual Salary'],bins=[0,60000,100000,150000,300000],labels=['<$60K','$60K–$100K','$100K–$150K','>$150K'])
df['Bonus Amount']    = (df['Annual Salary'] * df['Bonus %']).round(0)#to turn percentages into real money
#Drop rows missing critical fields
before = len(df)
df = df.dropna(subset=['Annual Salary','Department','Gender'])
after  = len(df)
print(f"Rows after cleaning: {after:,}  (removed {before - after} incomplete records)")
print(f"New derived columns added:")
print("['Tenure Years','Is Active','Hire Year','Age Band','Salary Band','Bonus Amount']")


Rows after cleaning: 987  (removed 13 incomplete records)
New derived columns added:
['Tenure Years','Is Active','Hire Year','Age Band','Salary Band','Bonus Amount']


## 4. Descriptive Statistics

In [7]:
print("=== Workforce Summary ===")
print(f"  Total employees (all time) : {len(df)}")
print(f"  Currently active           : {df['Is Active'].sum()}")
print(f"  Employees who left (not active): {(~df['Is Active']).sum():,}")
print(f"  the percentage of employees who left the company: {(~df['Is Active']).mean()*100:.1f}%")
print(f'\n')
print(f"=== Salary ===")
print(f"  Average Annual Salary      : ${df['Annual Salary'].mean():,.0f}")
print(f"  Median Annual Salary       : ${df['Annual Salary'].median():,.0f}")
print(f"  Min / Max                  : ${df['Annual Salary'].min():,.0f} / ${df['Annual Salary'].max():,.0f}")
print(f'\n')
print(f"=== Age & Tenure ===")
print(f"  Average Age                : {df['Age'].mean().round()} years")
print(f"  Average Tenure             : {df['Tenure Years'].mean().round()} years")
print(f'\n')
print(f"=== By Department ===")
dept_summary = df.groupby('Department').agg(Headcount=('EEID','count'),Avg_Salary=('Annual Salary','mean'),Avg_Age=('Age','mean'),Avg_Tenure=('Tenure Years','mean')).round()
dept_summary['Avg_Salary'] = dept_summary['Avg_Salary'].map('${:,.0f}'.format)
print(dept_summary.to_string())

=== Workforce Summary ===
  Total employees (all time) : 987
  Currently active           : 903
  Employees who left (not active): 84
  the percentage of employees who left the company: 8.5%


=== Salary ===
  Average Annual Salary      : $113,467
  Median Annual Salary       : $96,598
  Min / Max                  : $40,063 / $258,498


=== Age & Tenure ===
  Average Age                : 44.0 years
  Average Tenure             : 13.0 years


=== By Department ===
                 Headcount Avg_Salary  Avg_Age  Avg_Tenure
Department                                                
Accounting              96   $123,147     44.0        12.0
Engineering            154   $109,377     46.0        14.0
Finance                119   $123,208     45.0        13.0
Human Resources        122   $118,257     44.0        13.0
IT                     237    $97,960     44.0        14.0
Marketing              120   $129,663     43.0        12.0
Sales                  139   $111,225     44.0        13.0


In [9]:
dept_summary = df.groupby('Department').agg(
    Headcount=('EEID', 'count'),
    Exits=('Is Active', lambda x: (~x).sum())
)

print(dept_summary.to_string())

                 Headcount  Exits
Department                       
Accounting              96      7
Engineering            154     17
Finance                119      9
Human Resources        122     11
IT                     237     15
Marketing              120     15
Sales                  139     10


In [10]:
dept_summary['Attrition_Rate_%'] = (
    dept_summary['Exits'] / dept_summary['Headcount'] * 100
).round(1)#percentage of employees or customers lost over a specific period who are not replaced
print(dept_summary.to_string())

                 Headcount  Exits  Attrition_Rate_%
Department                                         
Accounting              96      7               7.3
Engineering            154     17              11.0
Finance                119      9               7.6
Human Resources        122     11               9.0
IT                     237     15               6.3
Marketing              120     15              12.5
Sales                  139     10               7.2


In [14]:
attrition_bu = df.groupby('Business Unit').agg(
    Headcount=('EEID', 'count'),
    Exits=('Is Active', lambda x: (~x).sum())
)

attrition_bu['Attrition_Rate_%'] = (
    attrition_bu['Exits'] / attrition_bu['Headcount'] * 100
).round(1)

print(attrition_bu.reset_index().to_string(index=False))

         Business Unit  Headcount  Exits  Attrition_Rate_%
             Corporate        236     19               8.1
         Manufacturing        263     20               7.6
Research & Development        226     27              11.9
   Speciality Products        262     18               6.9


In [16]:
attrition_age = df.groupby('Age Band').agg(
    Headcount=('EEID', 'count'),
    Exits=('Is Active', lambda x: (~x).sum())
)

attrition_age['Attrition_Rate_%'] = (
    attrition_age['Exits'] / attrition_age['Headcount'] * 100
).round(1)

print(attrition_age.reset_index().to_string(index=False))

Age Band  Headcount  Exits  Attrition_Rate_%
   25–34        242     23               9.5
   35–44        223     20               9.0
   45–54        291     26               8.9
   55–65        230     15               6.5


In [17]:
attrition_gender = df.groupby('Gender').agg(
    Headcount=('EEID', 'count'),
    Exits=('Is Active', lambda x: (~x).sum())
)

attrition_gender['Attrition_Rate_%'] = (
    attrition_gender['Exits'] / attrition_gender['Headcount'] * 100
).round(1)

print(attrition_gender.reset_index().to_string(index=False))

Gender  Headcount  Exits  Attrition_Rate_%
Female        511     39               7.6
  Male        476     45               9.5


In [21]:
job_risk = df.groupby('Job Title').agg(
    Headcount=('EEID', 'count'),
    Exits=('Is Active', lambda x: (~x).sum())
)

job_risk = job_risk[job_risk['Headcount'] >= 10]


job_risk['Attrition_Rate_%'] = (
    job_risk['Exits'] / job_risk['Headcount'] * 100
).round(1)
job_risk = job_risk[job_risk['Attrition_Rate_%'] >= 10]

job_risk = job_risk.sort_values(by='Attrition_Rate_%', ascending=False)

print(job_risk.reset_index().to_string(index=False))

             Job Title  Headcount  Exits  Attrition_Rate_%
      Business Partner         18      5              27.8
  Sr. Business Partner         16      3              18.8
   Operations Engineer         12      2              16.7
   Engineering Manager         20      3              15.0
Account Representative         21      3              14.3
 System Administrator          14      2              14.3
          HRIS Analyst         16      2              12.5
           Sr. Analyst         69      8              11.6
              Director        119     12              10.1
 Network Administrator         10      1              10.0
      Quality Engineer         20      2              10.0


In [31]:
attrition_ethnicity = df.groupby('Ethnicity').agg(
    Headcount=('EEID', 'count'),
    Exits=('Is Active', lambda x: (~x).sum())
)

attrition_ethnicity['Attrition_Rate_%'] = (
    attrition_ethnicity['Exits'] / attrition_ethnicity['Headcount'] * 100
).round(1)

print(attrition_ethnicity.reset_index().to_string(index=False))

Ethnicity  Headcount  Exits  Attrition_Rate_%
    Asian        400     38               9.5
    Black         73     10              13.7
Caucasian        263     18               6.8
   Latino        249     18               7.2


In [32]:
country_dist = df.groupby('Country').agg(
    Headcount=('EEID', 'count')
).sort_values(by='Headcount', ascending=False)
country_dist['Percentage_%'] = (
    country_dist['Headcount'] / len(df) * 100
).round(1)
print(country_dist.reset_index().to_string(index=False))

      Country  Headcount  Percentage_%
United States        632          64.0
        China        217          22.0
       Brazil        138          14.0


## 5. Key Visualisations

In [34]:
PRIMARY='#1B4F72'; SECONDARY='#2874A6'; ACCENT='#3498DB'; HIGHLIGHT='#E74C3C'
plt.rcParams.update({'font.family':'DejaVu Sans','axes.spines.top':False,'axes.spines.right':False})

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('HR Dashboard — Key Metrics', fontsize=16, fontweight='bold', y=1.01)

# 1 Headcount by dept
dept_c = df['Department'].value_counts()
axes[0,0].barh(dept_c.index, dept_c.values, color=SECONDARY)
axes[0,0].set_title('Headcount by Department'); axes[0,0].set_xlabel('Employees')

# 2 Avg salary by dept
avg_s = df.groupby('Department')['Annual Salary'].mean().sort_values()/1000
axes[0,1].barh(avg_s.index, avg_s.values, color=PRIMARY)
axes[0,1].set_title('Avg Salary by Dept ($K)'); axes[0,1].set_xlabel('$K')

# 3 Gender pie
g = df['Gender'].value_counts()
axes[0,2].pie(g.values, labels=g.index, autopct='%1.0f%%', colors=[ACCENT,HIGHLIGHT],
              wedgeprops={'edgecolor':'white','linewidth':2})
axes[0,2].set_title('Gender Split')

# 4 Ethnicity
e = df['Ethnicity'].value_counts()
axes[0,3].bar(e.index, e.values, color=[PRIMARY,SECONDARY,ACCENT,'#AED6F1'])
axes[0,3].set_title('Ethnicity'); axes[0,3].tick_params(axis='x',rotation=15)

# 5 Age histogram
axes[1,0].hist(df['Age'].dropna(), bins=15, color=ACCENT, edgecolor='white')
axes[1,0].axvline(df['Age'].mean(), color=HIGHLIGHT, linestyle='--', label=f'Mean {df["Age"].mean():.0f}')
axes[1,0].set_title('Age Distribution'); axes[1,0].legend(fontsize=8)

# 6 Hiring trend
hy = df.groupby('Hire Year').size()
axes[1,1].plot(hy.index, hy.values, color=PRIMARY, marker='o', linewidth=2)
axes[1,1].fill_between(hy.index, hy.values, alpha=0.15, color=ACCENT)
axes[1,1].set_title('Annual Hiring Trend'); axes[1,1].tick_params(axis='x',rotation=30)

# 7 Salary by gender boxplot
genders = df['Gender'].dropna().unique()
axes[1,2].boxplot([df[df['Gender']==g]['Annual Salary'].dropna()/1000 for g in genders],
                   labels=genders, patch_artist=True,
                   boxprops={'facecolor':ACCENT,'alpha':0.7},
                   medianprops={'color':'white','linewidth':2})
axes[1,2].set_title('Salary by Gender ($K)')

# 8 Country
cnt = df['Country'].value_counts()
axes[1,3].bar(cnt.index, cnt.values, color=[PRIMARY,SECONDARY,ACCENT])
axes[1,3].set_title('Global Distribution')

plt.tight_layout()
plt.savefig('dashboard_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Dashboard saved")

Dashboard saved


## 6. Export Clean Dataset

In [33]:
output_path = 'employee_clean.xlsx'
df.to_excel(output_path, index=False)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("Final columns:")
df.columns

Shape: 987 rows × 21 columns
Final columns:


Index(['EEID', 'Full Name', 'Job Title', 'Department', 'Business Unit',
       'Gender', 'Ethnicity', 'Age', 'Hire Date', 'Annual Salary', 'Bonus %',
       'Country', 'City', 'Exit Date', 'Tenure Years', 'Is Active',
       'Hire Year', 'Hire Quarter', 'Age Band', 'Salary Band', 'Bonus Amount'],
      dtype='object')